# EXPRESS Baseline Reproduction Runner

This notebook runs the paper-style baseline pipeline from this repo and writes artifacts to the repo top level.

Generated top-level artifacts:
- `baseline_raw_<model>.csv`
- `baseline_clean_<model>.csv`
- `baseline_metrics_<model>.json`
- `baseline_summary_<model>.md`


In [ ]:
from pathlib import Path
import os
import sys
import re
import json
import shlex
import subprocess
import pandas as pd

EXPECTED_ROOT = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora/express-emotion-recognition')
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = EXPECTED_ROOT
    os.chdir(REPO_ROOT)

SRC_DIR = REPO_ROOT / 'src'
DATA_DIR = REPO_ROOT / 'data'
RESULTS_DIR = REPO_ROOT / 'results'

MODEL_SCRIPT = SRC_DIR / 'model-inference.py'
CLEAN_SCRIPT = SRC_DIR / 'evaluation' / 'result-cleaning.py'
EVAL_SCRIPT = SRC_DIR / 'evaluation' / 'result-evaluation.py'
DATA_FILE = DATA_DIR / 'express.csv'
BREAKDOWN_FILE = DATA_DIR / 'lexicon-decomposition.csv'

for p in [MODEL_SCRIPT, CLEAN_SCRIPT, EVAL_SCRIPT, DATA_FILE, BREAKDOWN_FILE]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print('REPO_ROOT:', REPO_ROOT)
print('Python:', sys.executable)


In [ ]:
def run_cmd(cmd, cwd=REPO_ROOT, env=None, check=True):
    printable = ' '.join(shlex.quote(str(x)) for x in cmd)
    print(f'$ {printable}')
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd),
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    captured = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        captured.append(line)

    return_code = proc.wait()
    output_text = ''.join(captured)
    if check and return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}: {printable}')
    return output_text

def parse_eval_metrics(stdout_text):
    metrics = {}
    for key in ['VRate', 'AccL', 'AccV', 'F1V', 'AccV-2']:
        m = re.search(rf'{re.escape(key)}\s*=\s*([0-9]*\.?[0-9]+)', stdout_text)
        if m:
            metrics[key] = float(m.group(1))
    return metrics


In [ ]:
express_head = pd.read_csv(DATA_FILE, nrows=3)
num_rows = sum(1 for _ in open(DATA_FILE, 'r', encoding='utf-8', errors='ignore')) - 1

print('Dataset rows:', num_rows)
print('Dataset columns:', list(express_head.columns))
display(express_head)

existing_results = sorted(p.name for p in RESULTS_DIR.glob('*.csv'))
print('Existing baseline result files:')
for name in existing_results:
    print(' -', name)


## Existing Baseline Check

This runs the official evaluation script on an existing result file to verify the pipeline before fresh inference.


In [ ]:
existing_baseline_file = RESULTS_DIR / 'gemma2_2B_it.csv'
if not existing_baseline_file.exists():
    raise FileNotFoundError(f'Missing expected baseline file: {existing_baseline_file}')

eval_stdout_existing = run_cmd([
    sys.executable,
    EVAL_SCRIPT,
    '--result_path', existing_baseline_file,
    '--breakdowns_path', BREAKDOWN_FILE,
])

existing_metrics = parse_eval_metrics(eval_stdout_existing)
print('Parsed existing metrics:', existing_metrics)


## Fresh Baseline Run Settings

Set `RUN_INFERENCE=True` for a fresh run.
- Keep `FULL_DATA=False` for a quick sanity run.
- Set `FULL_DATA=True` to reproduce full baseline across all rows.


In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN', '')
MODEL_NAME = 'google/gemma-2-2b-it'
BATCH_SIZE = 4
RUN_INFERENCE = False
FULL_DATA = False
SAMPLE_ROWS = 256
DISABLE_8BIT = False

model_slug = MODEL_NAME.split('/')[-1].replace('-', '_').replace('.', '_')
raw_output = REPO_ROOT / f'baseline_raw_{model_slug}.csv'
clean_output = REPO_ROOT / f'baseline_clean_{model_slug}.csv'
metrics_output = REPO_ROOT / f'baseline_metrics_{model_slug}.json'
summary_output = REPO_ROOT / f'baseline_summary_{model_slug}.md'
inference_input = REPO_ROOT / f'baseline_input_{model_slug}.csv'
download_cache = REPO_ROOT / 'outputs' / 'hf_cache' / 'hf_cache'
download_cache.mkdir(exist_ok=True)

print('RUN_INFERENCE =', RUN_INFERENCE)
print('FULL_DATA =', FULL_DATA)
print('MODEL_NAME =', MODEL_NAME)
print('raw_output =', raw_output)
print('clean_output =', clean_output)
print('metrics_output =', metrics_output)



In [ ]:
if FULL_DATA:
    run_input_path = DATA_FILE
    print('Using full dataset as inference input:', run_input_path)
else:
    df_all = pd.read_csv(DATA_FILE)
    sample_n = min(SAMPLE_ROWS, len(df_all))
    df_sample = df_all.sample(n=sample_n, random_state=42).sort_index().reset_index(drop=True)
    df_sample.to_csv(inference_input, index=False)
    run_input_path = inference_input
    print(f'Wrote sample inference input ({sample_n} rows):', run_input_path)


In [ ]:
if RUN_INFERENCE:
    if not HF_TOKEN:
        raise ValueError('HF_TOKEN is empty. Set your Hugging Face token in environment before running inference.')

    inference_cmd = [
        sys.executable,
        MODEL_SCRIPT,
        '--model', MODEL_NAME,
        '--input', run_input_path,
        '--output', raw_output,
        '--download_path', download_cache,
        '--batch_size', str(BATCH_SIZE),
    ]

    if DISABLE_8BIT:
        inference_cmd.append('--disable_8bit')

    _ = run_cmd(inference_cmd, env={'HF_TOKEN': HF_TOKEN})
else:
    print('Inference skipped (RUN_INFERENCE=False).')


In [ ]:
if RUN_INFERENCE:
    _ = run_cmd([
        sys.executable,
        CLEAN_SCRIPT,
        '--input', raw_output,
        '--output', clean_output,
    ])
    evaluation_target = clean_output
else:
    evaluation_target = existing_baseline_file

print('Evaluation target:', evaluation_target)


In [ ]:
eval_stdout = run_cmd([
    sys.executable,
    EVAL_SCRIPT,
    '--result_path', evaluation_target,
    '--breakdowns_path', BREAKDOWN_FILE,
])

metrics = parse_eval_metrics(eval_stdout)
row_count = len(pd.read_csv(evaluation_target))

payload = {
    'model_name': MODEL_NAME,
    'evaluation_target': str(evaluation_target),
    'rows': int(row_count),
    'metrics': metrics,
    'run_inference': RUN_INFERENCE,
    'full_data': FULL_DATA,
    'sample_rows': int(SAMPLE_ROWS),
}

metrics_output.write_text(json.dumps(payload, indent=2), encoding='utf-8')

summary_lines = [
    f'# Baseline Summary ({model_slug})',
    '',
    f'- model_name: {MODEL_NAME}',
    f'- evaluation_target: {evaluation_target}',
    f'- rows: {row_count}',
    f'- run_inference: {RUN_INFERENCE}',
    f'- full_data: {FULL_DATA}',
    '',
]
for k, v in metrics.items():
    summary_lines.append(f'- {k}: {v}')

summary_output.write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')

print('Saved metrics JSON:', metrics_output)
print('Saved summary markdown:', summary_output)
print('Parsed metrics:', metrics)


## Quick Plots

Run this after the evaluation cell to visualize baseline quality.

In [ ]:
import ast
import matplotlib.pyplot as plt
import seaborn as sns

if "metrics" not in globals() or "evaluation_target" not in globals():
    raise RuntimeError("Run the evaluation cell above first.")

sns.set_theme(style="whitegrid")

metric_df = pd.DataFrame({"metric": list(metrics.keys()), "value": list(metrics.values())})
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=metric_df, x="metric", y="value", ax=axes[0], color="#4C78A8")
axes[0].set_ylim(0, 1)
axes[0].set_title(f"Baseline metrics ({model_slug})")
axes[0].tick_params(axis="x", rotation=30)
for i, v in enumerate(metric_df["value"]):
    axes[0].text(i, min(v + 0.02, 0.99), f"{v:.3f}", ha="center", fontsize=9)

def parse_list_cell(x):
    if isinstance(x, list):
        return [str(i).lower() for i in x]
    if isinstance(x, str):
        try:
            val = ast.literal_eval(x)
            if isinstance(val, list):
                return [str(i).lower() for i in val]
        except Exception:
            return []
    return []

eval_df = pd.read_csv(evaluation_target)
eval_df["labels_list"] = eval_df["labels"].apply(parse_list_cell)
eval_df["pred_list"] = eval_df["output_formatted"].apply(parse_list_cell)

mismatched_true_labels = []
for true_list, pred_list in zip(eval_df["labels_list"], eval_df["pred_list"]):
    for true_label, pred_label in zip(true_list, pred_list):
        if true_label != pred_label:
            mismatched_true_labels.append(true_label)

if mismatched_true_labels:
    top_errors = pd.Series(mismatched_true_labels).value_counts().head(12)
    sns.barplot(x=top_errors.values, y=top_errors.index, ax=axes[1], color="#E45756")
    axes[1].set_title("Most missed true emotions")
    axes[1].set_xlabel("Mismatch count")
    axes[1].set_ylabel("True label")
else:
    axes[1].text(0.5, 0.5, "No mismatches found", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()


## Reproduce Figure 2 (from data)

This cell recomputes **vector accuracy (AccV)** for each model from the CSV result files in `results/`, then plots model size vs AccV by family.

In [ ]:
import ast
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path.cwd()
results_dir = repo_root / "results"
lexicon_path = repo_root / "data" / "lexicon-decomposition.csv"

lex_df = pd.read_csv(lexicon_path)
vec_map = {
    str(row["word"]).lower(): tuple(int(v) for v in row.iloc[1:].tolist())
    for _, row in lex_df.iterrows()
}
zero_vec = tuple([0] * 10)

parse_cache = {}
def parse_list_cached(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if s == "INVALID OUTPUT":
        return None
    if s in parse_cache:
        return parse_cache[s]
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            parsed = [str(x).strip().lower() for x in val]
        else:
            parsed = None
    except Exception:
        parsed = None
    parse_cache[s] = parsed
    return parsed

def compute_accv(csv_path):
    df = pd.read_csv(csv_path, usecols=["labels", "output_formatted", "number_of_labels"])
    total_masks = int(df["number_of_labels"].sum())
    matches = 0
    for raw_labels, raw_preds in zip(df["labels"].tolist(), df["output_formatted"].tolist()):
        labels = parse_list_cached(raw_labels)
        preds = parse_list_cached(raw_preds)
        if labels is None or preds is None:
            continue
        for t, p in zip(labels, preds):
            matches += int(vec_map.get(t, zero_vec) == vec_map.get(p, zero_vec))
    return (matches / total_masks) if total_masks > 0 else float("nan")

# Size values (in billions of parameters) used in the paper discussion.
models = [
    {"label": "Flan-T5-large", "family": "Flan-T5", "size_b": 0.78, "file": "flan_t5_large.csv"},
    {"label": "Flan-T5-xl", "family": "Flan-T5", "size_b": 3.0, "file": "flan_t5_xl.csv"},
    {"label": "Flan-T5-xxl", "family": "Flan-T5", "size_b": 11.0, "file": "flan_t5_xxl.csv"},
    {"label": "Gpt-3.5-turbo", "family": "GPT", "size_b": 175.0, "file": "gpt_35_turbo_0125.csv"},
    {"label": "Gpt-4o", "family": "GPT", "size_b": 1750.0, "file": "gpt_4o.csv"},
    {"label": "Gemma-2-2B-it", "family": "Gemma2", "size_b": 2.0, "file": "gemma2_2B_it.csv"},
    {"label": "Gemma-2-9B-it", "family": "Gemma2", "size_b": 9.0, "file": "gemma2_9B_it.csv"},
    {"label": "Gemma-2-27B-it", "family": "Gemma2", "size_b": 27.0, "file": "gemma2_27B_it.csv"},
    {"label": "Llama3.1-8B-instruct", "family": "Llama3.1", "size_b": 8.0, "file": "llama3.1_8B_instruct.csv"},
    {"label": "Llama3.1-70B-instruct", "family": "Llama3.1", "size_b": 70.0, "file": "llama3.1_70B_instruct.csv"},
    {"label": "Longformer-base", "family": "Longformer", "size_b": 0.149, "file": "longformer.csv"},
    {"label": "Mental-Longformer-base", "family": "Longformer", "size_b": 0.149, "file": "mental-longformer.csv"},
    {"label": "Roberta-base", "family": "Roberta", "size_b": 0.125, "file": "roberta-base.csv"},
    {"label": "Mental-Roberta-base", "family": "Roberta", "size_b": 0.125, "file": "mental-roberta-base.csv"},
]

rows = []
for m in models:
    path = results_dir / m["file"]
    if not path.exists():
        print(f"Skipping missing file: {path}")
        continue
    print(f"Computing AccV for {m['label']} from {m['file']} ...")
    accv = compute_accv(path)
    rows.append({**m, "accv": accv})

fig2_df = pd.DataFrame(rows)
fig2_df = fig2_df.sort_values(["family", "size_b"]).reset_index(drop=True)
display(fig2_df[["label", "family", "size_b", "accv"]])

sns.set_theme(style="whitegrid")
family_colors = {
    "Flan-T5": "#4C72B0",
    "GPT": "#DD8452",
    "Gemma2": "#55A868",
    "Llama3.1": "#C44E52",
    "Longformer": "#8172B2",
    "Roberta": "#937860",
}
family_markers = {
    "Flan-T5": "D",
    "GPT": "P",
    "Gemma2": "^",
    "Llama3.1": "X",
    "Longformer": "s",
    "Roberta": "o",
}

plt.figure(figsize=(14, 6))
for fam in ["Flan-T5", "GPT", "Gemma2", "Llama3.1", "Longformer", "Roberta"]:
    sub = fig2_df[fig2_df["family"] == fam].sort_values("size_b")
    if sub.empty:
        continue
    plt.plot(
        sub["size_b"],
        sub["accv"],
        marker=family_markers[fam],
        color=family_colors[fam],
        linestyle="--",
        linewidth=1.25,
        markersize=8,
        label=fam,
    )
    for _, r in sub.iterrows():
        plt.text(r["size_b"] * 0.96, r["accv"] + 0.003, r["label"], fontsize=10)

plt.xscale("log")
plt.xlabel("Model Size")
plt.ylabel("Accuracy (Vector)")
plt.ylim(0.08, 0.40)
plt.xticks([0.1, 1, 10, 100, 1000], ["100M", "1B", "10B", "100B", "1000B"])
plt.legend(title="Model Family", loc="lower left")
plt.title("Figure 2 Reproduction: Model Size vs Vector Accuracy (AccV)")
plt.tight_layout()

fig2_path = repo_root / "figure2_reproduced_from_data.png"
plt.savefig(fig2_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {fig2_path}")
